In [ ]:
import pandas as pd
import sys
import os
import yaml
from pathlib import Path

DATA_DIR = ''

In [ ]:
# scenario.config and test.xml paths (available even if read_output_files fails)
test_dirs = list(Path(DATA_DIR).rglob("test.xml"))
test_dir = test_dirs[0].parent if test_dirs else Path(DATA_DIR)
test_xml_path = test_dir / "test.xml"
scenario_config_path = test_dir.parent / "_config" / "scenario.config"

# Load scenario.config for map_file, n_waypoints, etc. (for display when CSV data fails)
scenario_config = {}
if scenario_config_path.exists():
    with open(scenario_config_path) as f:
        cfg = yaml.safe_load(f)
        scenario_config = next(iter(cfg.values())) if isinstance(cfg, dict) and len(cfg) == 1 else (cfg or {})
map_file = scenario_config.get("map_file") if scenario_config else None
config_dir = test_dir.parent.name

In [ ]:
# Two-column table: test.xml | scenario.config
import textwrap
import xml.etree.ElementTree as ET
from datetime import datetime
from IPython.display import display, HTML

test_xml_lines = []
if test_xml_path.exists():
    try:
        tree = ET.parse(test_xml_path)
        root = tree.getroot()
        suite = root if root.tag == "testsuite" else root.find("testsuite")
        if suite is not None:
            errors = int(suite.get("errors", 0))
            failures = int(suite.get("failures", 0))
            ok = errors == 0 and failures == 0
            duration = float(suite.get("time", 0))
            tc = suite.find("testcase")
            start_time_iso = ""
            if tc is not None:
                for prop in tc.findall("./properties/property[@name='start_time']"):
                    ts = float(prop.get("value", 0))
                    start_time_iso = datetime.fromtimestamp(ts).isoformat()
                    break
            test_xml_lines.append("✓ Success" if ok else "✗ Failure")
            test_xml_lines.append(f"Duration: {duration:.6g} s")
            if start_time_iso:
                test_xml_lines.append(f"Start time: {start_time_iso}")
            if not ok and tc is not None:
                for elem in list(tc.findall("failure")) + list(tc.findall("error")):
                    msg = elem.get("message", "")
                    body = (elem.text or "").strip()
                    if msg:
                        test_xml_lines.append(f"Failure: {msg}")
                    if body:
                        for para in body.splitlines():
                            for line in textwrap.wrap(para, width=100) or [""]:
                                test_xml_lines.append(line)
    except Exception as e:
        test_xml_lines.append(f"Error parsing: {e}")
else:
    test_xml_lines.append("test.xml not found")

scenario_text = scenario_config_path.read_text() if scenario_config_path.exists() else "scenario.config not found"

display(HTML(
    '<div style="display: grid; grid-template-columns: 2fr 2fr; gap: 1em; font-family: monospace; font-size: 0.9em;">'
    '<div><strong>Test Result</strong><pre style="margin-top: 0.5em;">' + "\n".join(test_xml_lines) + '</pre></div>'
    '<div><strong>Scenario Config</strong><pre style="margin-top: 0.5em;">' + scenario_text.replace("<", "&lt;").replace(">", "&gt;") + '</pre></div>'
    '</div>'
))

In [ ]:
from robovast.common.analysis import (CampaignDataError, config_file, get_behavior_info,
                                      list_tables, read_table)
# Public, but not re-exported from the package, so it comes from the submodule. Imported
# rather than reimplemented: which clock a pose may be differentiated against is the
# library's rule, and a second copy of it here is a copy that goes stale.
from robovast.common.analysis.metrics import pose_time_base

# These notebooks serve every .vast in this directory, and those split into two families:
# five scenarios drive `robot.follow_waypoints`, and scenario_nav_through_poses.osc drives
# scenario_nav_through_poses.osc drive `robot.nav_through_poses`. The behaviour is
# recorded under `<actor type>.<action>`, so a name hardcoded to either family leaves the
# other with empty duration panels and no error -- read it from the tree instead.
NAV_ACTIONS = ('nav_through_poses', 'follow_waypoints', 'nav_to_pose')

# Whichever navigator the scenario called wrote its own feedback table, and a campaign has
# only that one. Both carry distance_remaining and number_of_recoveries, which is what the
# panels below plot.
ACTION_FEEDBACK_TABLES = ('action_navigate_through_poses_feedback',
                          'action_navigate_to_pose_feedback')


def navigation_behavior(behaviors):
    """The navigation behaviour's name as this campaign recorded it."""
    names = set(behaviors['behavior_name'].dropna())
    for action in NAV_ACTIONS:
        found = sorted(n for n in names if n.rsplit('.', 1)[-1] == action)
        if found:
            return found[0]
    raise SystemExit(
        f"No navigation behaviour here. Looked for {', '.join(NAV_ACTIONS)}; the tree "
        f"recorded: {', '.join(sorted(n for n in names if '.' in n)) or '(none)'}.")


def action_feedback(data_dir):
    """The navigator's feedback frame, or None when this campaign recorded none."""
    present = [t for t in ACTION_FEEDBACK_TABLES if t in list_tables(data_dir)]
    if not present:
        print("No navigation-action feedback table: the recovery and distance-remaining "
              "panels below are omitted.")
        return None
    print(f"Action feedback from {present[0]}.")
    return read_table(data_dir, present[0])


# The poses are what every panel below is drawn from, so a run without them has nothing to
# show. Said once, here, rather than as a NameError six cells down -- the Explorer renders
# outputs only, so a traceback there names a variable and not the reason.
# One read per table, so the message names the one that is missing. Together in a single
# try, a campaign with poses but no behaviour tree -- execution.bt_log off, or an execution
# image whose scenario_execution predates --bt-log -- reported "no postprocessed data",
# which is a different problem with a different fix.
try:
    df = read_table(DATA_DIR, "poses", with_params=True,
                    require=['frame', 'timestamp', 'position.x', 'position.y',
                             'orientation.yaw'])
except CampaignDataError as e:
    print(e)
    raise SystemExit("This run has no postprocessed data to analyse.")

try:
    df_behaviors = read_table(DATA_DIR, "behaviors",
                              require=['behavior_name', 'status_name', 'timestamp'])
except CampaignDataError as e:
    # Every panel from here on is keyed off the navigation behaviour, so there is no
    # partial view worth rendering -- but the reason is not the one above.
    print(e)
    raise SystemExit(
        "This run recorded no behaviour tree, so its navigation cannot be timed or "
        "named. See the message above for the two ways that happens.")

if df.empty:
    raise SystemExit("This run recorded no poses.")

# `timestamp` is when the message ARRIVED and stays the join key; `stamp` is when the pose
# was true, and is the only one an interval may be computed from.
TIME = pose_time_base(df)
if TIME != 'stamp':
    print("No publisher timestamps in this campaign: intervals below are arrival times, "
          "quantized by /clock and jittered by delivery.")

NAV_BEHAVIOR = navigation_behavior(df_behaviors)
print(f"Navigation behaviour: {NAV_BEHAVIOR}.")
df_behavior_info = get_behavior_info(NAV_BEHAVIOR, df_behaviors)

action_feedback_df = action_feedback(DATA_DIR)

In [ ]:
# create statistics from the dataframes
import numpy as np
from robovast.common.analysis import calculate_speeds_from_poses

def traveled_distance(poses_df):
    """Sum of consecutive Euclidean distances along the path (in meters)."""
    if poses_df is None or len(poses_df) < 2:
        return np.nan
    x = poses_df['position.x'].values
    y = poses_df['position.y'].values
    dx = np.diff(x)
    dy = np.diff(y)
    return np.sum(np.sqrt(dx**2 + dy**2))

stats = {}
df_robot = df[df['frame'] == 'base_link'].copy()
df_groundtruth = df[df['frame'] == 'nav2_turtlebot4_base_link_gt'].copy()
if df_groundtruth.empty:
    df_groundtruth = df[df['frame'] == 'turtlebot4_base_link_gt'].copy()

# Ordered by the measurement clock: a path summed in arrival order double-counts wherever
# two samples landed in one tick and came back out of order.
df_robot_sorted = df_robot.sort_values(TIME)
for (run, config), group in df_robot_sorted.groupby(['run_id', 'config_name'], observed=False):
    d = traveled_distance(group)
    key = config if config else str(run)
    stats.setdefault(key, {})['traveled_distance_m'] = d

# speed (min, average, max) from ground truth or robot
df_speeds = calculate_speeds_from_poses(df_robot_sorted)
if not df_speeds.empty:
    for (run, config), group in df_speeds.groupby(['run_id', 'config_name'], observed=False):
        key = config if config else str(run)
        stats.setdefault(key, {}).update({
            'speed_min_m_s': group['linear_speed'].min(),
            'speed_avg_m_s': group['linear_speed'].mean(),
            'speed_max_m_s': group['linear_speed'].max(),
        })

if action_feedback_df is not None and not action_feedback_df.empty:
    reco_col = 'feedback_number_of_recoveries'
    if reco_col in action_feedback_df.columns:
        for (run, config), group in action_feedback_df.groupby(['run_id', 'config_name'], observed=False):
            key = config if config else str(run)
            n_recoveries = group[reco_col].max()
            stats.setdefault(key, {})['recoveries'] = int(n_recoveries)

# display
from IPython.display import display, HTML
if stats:
    rows = []
    for config_key, s in stats.items():
        row = [config_key]
        row.append(f"{s.get('traveled_distance_m', '—'):.2f}" if isinstance(s.get('traveled_distance_m'), (int, float)) else str(s.get('traveled_distance_m', '—')))
        row.append(f"{s.get('speed_avg_m_s', '—'):.3f}" if isinstance(s.get('speed_avg_m_s'), (int, float)) else str(s.get('speed_avg_m_s', '—')))
        row.append(f"{s.get('speed_max_m_s', '—'):.3f}" if isinstance(s.get('speed_max_m_s'), (int, float)) else str(s.get('speed_max_m_s', '—')))
        row.append(str(s.get('recoveries', '—')))
        rows.append(row)
    headers = ['config', 'Traveled (m)', 'Speed avg (m/s)', 'Speed max (m/s)', 'Recoveries']
    html = '<table border="1" cellpadding="4" style="border-collapse: collapse;"><tr>' + ''.join(f'<th>{h}</th>' for h in headers) + '</tr>'
    for r in rows:
        html += '<tr>' + ''.join(f'<td>{c}</td>' for c in r) + '</tr>'
    html += '</table>'
    display(HTML(html))
else:
    print("No statistics: this run's poses produced neither a path nor a speed.")

In [ ]:
# `robovast_nav.map_visualizer`, not `robovast_nav.gui`: the gui package went with the Qt
# desktop tools, and the matplotlib visualizer moved up beside its siblings.
from robovast_nav.map_visualizer import MapVisualizer
from robovast.common.analysis import get_scenario_parameter
from IPython.display import HTML, display
from pathlib import Path
import io
import base64

viz = MapVisualizer()

_map_file = df['map_file'].iloc[0]
_config_dir = df['config_name'].iloc[0]
map_path = config_file(DATA_DIR, _map_file, _config_dir, must_exist=False) if _map_file else None

map_loaded = map_path and os.path.exists(map_path) and viz.load_map(map_path)
if not map_loaded:
    print(f"Map not loaded: {map_path or 'no map_file parameter'}. "
          f"The paths below are plotted without it.")
if map_loaded:
    viz.create_figure(figsize=(10, 10))
df_robot_mask = df['frame'] == 'base_link'
df_gt_mask = df['frame'] == 'nav2_turtlebot4_base_link_gt'
path_robot = list(zip(df.loc[df_robot_mask, 'position.x'], df.loc[df_robot_mask, 'position.y'], df.loc[df_robot_mask, 'orientation.yaw']))
path_groundtruth = list(zip(df.loc[df_gt_mask, 'position.x'], df.loc[df_gt_mask, 'position.y'], df.loc[df_gt_mask, 'orientation.yaw']))

map_img_b64 = None
if map_loaded:
    if path_groundtruth:
        viz.draw_path(path_groundtruth, linewidth=1, color='blue', show_endpoints=False)
    if path_robot:
        _ = viz.draw_path(path_robot, linewidth=1, color='red', show_endpoints=False)
    # Only the scenarios that place obstacles have this parameter; None means there are
    # none to draw, which is not the same as a failure to draw them.
    config_dir_local = os.path.dirname(DATA_DIR)
    _obstacles = get_scenario_parameter(config_dir_local, 'static_objects')
    if _obstacles:
        viz.draw_obstacles(_obstacles)
    if action_feedback_df is not None and not action_feedback_df.empty:
        af_for_viz = action_feedback_df[action_feedback_df['config_name'] == _config_dir]
        if len(af_for_viz) > 0:
            prev_recoveries = af_for_viz['feedback_number_of_recoveries'].shift(1).fillna(0)
            recovery_rows = af_for_viz[af_for_viz['feedback_number_of_recoveries'] > prev_recoveries]
            recovery_positions = []
            for _, row in recovery_rows.iterrows():
                x = row.get('feedback_current_pose_pose_position_x')
                y = row.get('feedback_current_pose_pose_position_y')
                if pd.notna(x) and pd.notna(y):
                    recovery_positions.append((float(x), float(y)))
                else:
                    pose = row.get('feedback_current_pose')
                    if isinstance(pose, dict) and 'pose' in pose and 'position' in pose['pose']:
                        p = pose['pose']['position']
                        recovery_positions.append((float(p['x']), float(p['y'])))
            if recovery_positions:
                viz.draw_waypoints(recovery_positions, color='red', marker='x', markersize=10, label='Recoveries')

    # draw start pose from scenario.config
    _sp = scenario_config.get('start_pose') if scenario_config else None
    if _sp:
        viz.add_robot_pose(
            _sp['position']['x'], _sp['position']['y'],
            _sp.get('orientation', {}).get('yaw', 0.0),
            color='green'
        )

    # The goal, however the scenario states it: `goal_pose` for a single-goal run,
    # `goal_poses` for the nav_through_poses and follow_waypoints families. Reading only
    # the singular drew no goal at all for either of those -- which is most of this
    # directory.
    _goals = scenario_config.get('goal_poses') if scenario_config else None
    if not _goals:
        _gp = scenario_config.get('goal_pose') if scenario_config else None
        _goals = [_gp] if _gp else []
    for _goal in _goals:
        viz.add_robot_pose(
            _goal['position']['x'], _goal['position']['y'],
            _goal.get('orientation', {}).get('yaw', 0.0),
            color='orange'
        )

    # also draw actual end pose from path_groundtruth
    if path_groundtruth:
        end_pose = path_groundtruth[-1]
        viz.add_robot_pose(end_pose[0], end_pose[1], end_pose[2], color='blue')

    buf = io.BytesIO()
    viz.fig.savefig(buf, format='png', dpi=100, bbox_inches='tight')
    import matplotlib.pyplot as plt
    plt.close(viz.fig)
    buf.seek(0)
    map_img_b64 = base64.b64encode(buf.read()).decode()

# Build layout: map | video (or just video if map unavailable)
video_filename = 'rosbag2_static_camera_image_compressed.webm'
video_path = os.path.join(DATA_DIR, video_filename)
video_html = ''
if os.path.exists(video_path):
    abs_path = os.path.abspath(video_path)
    file_url = Path(abs_path).as_uri()
    video_html = f'''<video controls loop autoplay muted style="width: 100%; height: 100%; object-fit: contain;">
        <source src="{file_url}" type="video/webm">
        Your browser does not support the video tag.
    </video>'''
else:
    video_html = f'<p>Video not found at: {video_path}</p>'

if map_img_b64:
    html = f'''<div style="display: flex; gap: 1em; align-items: flex-start;">
        <div style="flex: 1; min-width: 0;"><img src="data:image/png;base64,{map_img_b64}" style="width: 100%; height: auto;"></div>
        <div style="flex: 1; min-width: 0; min-height: 300px;">{video_html}</div>
    </div>'''
else:
    html = f'''<div style="display: flex; gap: 1em; align-items: flex-start;">
        <div style="flex: 1; min-width: 0; padding: 1em;">Map not available: see the message above.</div>
        <div style="flex: 1; min-width: 0; min-height: 300px;">{video_html}</div>
    </div>'''
display(HTML(html))

## Errors

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.interpolate import interp1d

# Split the dataframe by frame - use boolean indexing once and cache results
df_robot = df[df['frame'] == 'base_link'].copy()
df_groundtruth = df[df['frame'] == 'nav2_turtlebot4_base_link_gt'].copy()

# Sorted and interpolated on TIME, the measurement clock. Ordering by arrival time leaves
# rows within one tick in arbitrary order, which is enough to hand interp1d a
# non-monotonic x.
df_robot.sort_values(TIME, inplace=True)
df_robot.reset_index(drop=True, inplace=True)
df_groundtruth.sort_values(TIME, inplace=True)
df_groundtruth.reset_index(drop=True, inplace=True)

In [ ]:
# Find common time range - data already sorted from previous cell
if not df_robot.empty and not df_groundtruth.empty:
    time_start = max(df_robot[TIME].iloc[0], df_groundtruth[TIME].iloc[0])
    time_end = min(df_robot[TIME].iloc[-1], df_groundtruth[TIME].iloc[-1])

    # print(f"Common time range: {time_start:.2f} to {time_end:.2f} seconds")

    # Convert to numpy arrays for faster interpolation
    gt_timestamps = df_groundtruth[TIME].values
    gt_pos_x = df_groundtruth['position.x'].values
    gt_pos_y = df_groundtruth['position.y'].values

    # Create interpolation functions for ground truth
    gt_interp_x = interp1d(gt_timestamps, gt_pos_x, kind='linear', fill_value='extrapolate', assume_sorted=True)
    gt_interp_y = interp1d(gt_timestamps, gt_pos_y, kind='linear', fill_value='extrapolate', assume_sorted=True)

    gt_yaw = df_groundtruth['orientation.yaw'].values
    gt_interp_z = interp1d(gt_timestamps, gt_yaw, kind='linear', fill_value='extrapolate', assume_sorted=True)

    # Filter robot data to common time range - use boolean indexing efficiently
    time_mask = (df_robot[TIME] >= time_start) & (df_robot[TIME] <= time_end)
    df_robot_aligned = df_robot[time_mask].copy()

    # Convert to numpy array for vectorized operations
    robot_timestamps = df_robot_aligned[TIME].values

    # Create relative time (normalized to start at 0)
    df_robot_aligned['relative_time'] = robot_timestamps - robot_timestamps[0]

    # Interpolate ground truth at robot timestamps - batch operation
    df_robot_aligned['gt_x'] = gt_interp_x(robot_timestamps)
    df_robot_aligned['gt_y'] = gt_interp_y(robot_timestamps)

    df_robot_aligned['gt_z'] = gt_interp_z(robot_timestamps)
    
    # Calculate position errors - use numpy arrays for speed
    robot_x = df_robot_aligned['position.x'].values
    robot_y = df_robot_aligned['position.y'].values
    gt_x = df_robot_aligned['gt_x'].values
    gt_y = df_robot_aligned['gt_y'].values

    position_error_x = robot_x - gt_x
    position_error_y = robot_y - gt_y
    absolute_position_error = np.sqrt(position_error_x**2 + position_error_y**2)

    # Calculate orientation error if available
    robot_yaw = df_robot_aligned['orientation.yaw'].values
    gt_z = df_robot_aligned['gt_z'].values
    orientation_error = robot_yaw - gt_z
    # Normalize to [-pi, pi] - vectorized
    orientation_error = np.arctan2(np.sin(orientation_error), np.cos(orientation_error))
else:
    print("No aligned poses: this run recorded no base_link or no ground-truth frame, "
          "so the localization-error panels below are omitted.")
    df_robot_aligned = None

In [ ]:
if df_robot_aligned is not None and not df_robot_aligned.empty:
    # Create figure with subplots for errors
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))

    # Use numpy arrays directly for plotting (faster than series)
    timestamps = df_robot_aligned[TIME].values

    # 1. Absolute Position Error over time
    axes[0, 0].plot(timestamps, absolute_position_error, 'b-', linewidth=1)
    axes[0, 0].set_xlabel(f'Time (s, {TIME})')
    axes[0, 0].set_ylabel('Absolute Position Error (m)')
    axes[0, 0].set_title('Absolute Position Error Over Time')
    axes[0, 0].grid(True, alpha=0.3)

    # 2. Position Error Components (X and Y)
    axes[0, 1].plot(timestamps, position_error_x, 'r-', label='X Error', linewidth=1)
    axes[0, 1].plot(timestamps, position_error_y, 'g-', label='Y Error', linewidth=1)
    axes[0, 1].set_xlabel(f'Time (s, {TIME})')
    axes[0, 1].set_ylabel('Position Error (m)')
    axes[0, 1].set_title('Position Error Components')
    axes[0, 1].legend()
    axes[0, 1].grid(True, alpha=0.3)

    # 3. Absolute Orientation Error over time
    if orientation_error is not None:
        axes[1, 0].plot(timestamps, np.abs(orientation_error), 'purple', linewidth=1)
        axes[1, 0].set_xlabel(f'Time (s, {TIME})')
        axes[1, 0].set_ylabel('Absolute Orientation Error (rad)')
        axes[1, 0].set_title('Absolute Orientation Error Over Time')
        axes[1, 0].grid(True, alpha=0.3)
    else:
        axes[1, 0].text(0.5, 0.5, 'Orientation data not available', 
                        ha='center', va='center', transform=axes[1, 0].transAxes)
        axes[1, 0].set_title('Absolute Orientation Error Over Time')

    # 4. Relative Position Error (error between consecutive points)
    relative_position_error = np.diff(absolute_position_error)
    axes[1, 1].plot(timestamps[1:], relative_position_error, 'orange', linewidth=1)
    axes[1, 1].set_xlabel(f'Time (s, {TIME})')
    axes[1, 1].set_ylabel('Relative Position Error Change (m)')
    axes[1, 1].set_title('Relative Position Error (Rate of Change)')
    axes[1, 1].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()
else:
    absolute_position_error = None

In [ ]:
if absolute_position_error is not None:
    # Additional detailed analysis
    fig2, axes2 = plt.subplots(1, 2, figsize=(14, 5))

    # Pre-calculate statistics to avoid repeated computation
    mean_error = absolute_position_error.mean()
    median_error = np.median(absolute_position_error)

    # Distribution of absolute position errors
    axes2[0].hist(absolute_position_error, bins=50, color='blue', alpha=0.7, edgecolor='black')
    axes2[0].axvline(mean_error, color='red', linestyle='--', 
                    linewidth=2, label=f'Mean: {mean_error:.3f} m')
    axes2[0].axvline(median_error, color='green', linestyle='--', 
                    linewidth=2, label=f'Median: {median_error:.3f} m')
    axes2[0].set_xlabel('Absolute Position Error (m)')
    axes2[0].set_ylabel('Frequency')
    axes2[0].set_title('Distribution of Absolute Position Errors')
    axes2[0].legend()
    axes2[0].grid(True, alpha=0.3)

    # Cumulative distribution - optimized using numpy
    sorted_errors = np.sort(absolute_position_error)
    n = len(sorted_errors)
    cumulative = np.arange(1, n + 1) * (100.0 / n)
    axes2[1].plot(sorted_errors, cumulative, 'b-', linewidth=2)
    axes2[1].set_xlabel('Absolute Position Error (m)')
    axes2[1].set_ylabel('Cumulative Percentage (%)')
    axes2[1].set_title('Cumulative Distribution of Position Errors')
    axes2[1].grid(True, alpha=0.3)

    # Calculate percentiles once
    percentiles = [50, 90, 95]
    percentile_values = np.percentile(absolute_position_error, percentiles)

    # Mark specific percentiles
    for percentile, value in zip(percentiles, percentile_values):
        axes2[1].axhline(percentile, color='red', linestyle='--', alpha=0.5)
        axes2[1].axvline(value, color='red', linestyle='--', alpha=0.5)
        axes2[1].text(value, percentile + 2, f'{percentile}%: {value:.3f}m', fontsize=9)

    plt.tight_layout()
    plt.show()

    # Print statistics - use pre-calculated values where possible
    std_error = absolute_position_error.std()
    max_error = absolute_position_error.max()

    print("\n=== Position Error Statistics ===")
    print(f"Mean: {mean_error:.4f} m")
    print(f"Std Dev: {std_error:.4f} m")
    print(f"Median: {median_error:.4f} m")
    print(f"95th percentile: {percentile_values[2]:.4f} m")
    print(f"Max: {max_error:.4f} m")

    if orientation_error is not None:
        abs_orient_error = np.abs(orientation_error)
        mean_abs_orient = abs_orient_error.mean()
        std_orient = orientation_error.std()
        max_abs_orient = abs_orient_error.max()
        
        print("\n=== Orientation Error Statistics ===")
        print(f"Mean absolute: {mean_abs_orient:.4f} rad ({np.degrees(mean_abs_orient):.2f}°)")
        print(f"Std Dev: {std_orient:.4f} rad ({np.degrees(std_orient):.2f}°)")
        print(f"Max absolute: {max_abs_orient:.4f} rad ({np.degrees(max_abs_orient):.2f}°)")


## Speed Analysis

In [ ]:
df_gt_speeds = calculate_speeds_from_poses(df_groundtruth)

In [ ]:
# Create crisp speed visualizations
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# The clock the speeds beside it were differentiated from: calculate_speeds_from_poses
# returns both, and plotting one against the other would misplace every sample.
timestamps_gt = df_gt_speeds[TIME].values

# 1. Linear Speed Distribution
axes[0].hist(df_gt_speeds['linear_speed'].values, bins=40, color='steelblue', 
                edgecolor='black', linewidth=0.8, alpha=0.7)
mean_linear = df_gt_speeds['linear_speed'].mean()
median_linear = df_gt_speeds['linear_speed'].median()
axes[0].axvline(mean_linear, color='red', linestyle='--', linewidth=1.5, label=f'Mean: {mean_linear:.3f} m/s')
axes[0].axvline(median_linear, color='green', linestyle='--', linewidth=1.5, label=f'Median: {median_linear:.3f} m/s')
axes[0].set_xlabel('Linear Speed (m/s)', fontsize=10)
axes[0].set_ylabel('Frequency', fontsize=10)
axes[0].set_title('Linear Speed Distribution', fontsize=11, fontweight='bold')
axes[0].legend(fontsize=9)
axes[0].grid(True, alpha=0.3, linewidth=0.5, axis='y')

# 2. Angular Speed Distribution
axes[1].hist(df_gt_speeds['angular_speed'].values, bins=40, color='coral', 
                edgecolor='black', linewidth=0.8, alpha=0.7)
mean_angular = df_gt_speeds['angular_speed'].mean()
median_angular = df_gt_speeds['angular_speed'].median()
axes[1].axvline(mean_angular, color='red', linestyle='--', linewidth=1.5, label=f'Mean: {mean_angular:.3f} rad/s')
axes[1].axvline(median_angular, color='green', linestyle='--', linewidth=1.5, label=f'Median: {median_angular:.3f} rad/s')
axes[1].axvline(0, color='black', linestyle='-', linewidth=0.5)
axes[1].set_xlabel('Angular Speed (rad/s)', fontsize=10)
axes[1].set_ylabel('Frequency', fontsize=10)
axes[1].set_title('Angular Speed Distribution', fontsize=11, fontweight='bold')
axes[1].legend(fontsize=9)
axes[1].grid(True, alpha=0.3, linewidth=0.5, axis='y')

plt.tight_layout()
plt.show()

In [ ]:
# Combined speed profile visualization
fig, ax = plt.subplots(figsize=(14, 5))

# Create twin axis for angular speed
ax2 = ax.twinx()

# Plot linear speed on primary axis
line1 = ax.plot(timestamps_gt, df_gt_speeds['linear_speed'].values, 
                'b-', linewidth=1.5, label='Linear Speed', alpha=0.8)

# Plot angular speed on secondary axis
line2 = ax2.plot(timestamps_gt, df_gt_speeds['angular_speed'].values, 
                 'r-', linewidth=1.5, label='Angular Speed', alpha=0.8)

# Configure axes
ax.set_xlabel(f'Time (s, {TIME})', fontsize=11, fontweight='bold')
ax.set_ylabel('Linear Speed (m/s)', fontsize=11, fontweight='bold', color='b')
ax2.set_ylabel('Angular Speed (rad/s)', fontsize=11, fontweight='bold', color='r')
ax.set_title('Combined Speed Profile', fontsize=12, fontweight='bold')

# Style the axes
ax.tick_params(axis='y', labelcolor='b')
ax2.tick_params(axis='y', labelcolor='r')
ax.grid(True, alpha=0.3, linewidth=0.5)
ax.set_ylim(bottom=0)

# Add legend
lines = line1 + line2
labels = [l.get_label() for l in lines]
ax.legend(lines, labels, loc='upper right', fontsize=10)

plt.tight_layout()
plt.show()

In [ ]:
print(f"Ground truth speed statistics:")
print(f"Linear speed - Mean: {df_gt_speeds['linear_speed'].mean():.3f} m/s, Max: {df_gt_speeds['linear_speed'].max():.3f} m/s")
print(f"Angular speed - Mean abs: {np.abs(df_gt_speeds['angular_speed']).mean():.3f} rad/s, Max abs: {np.abs(df_gt_speeds['angular_speed']).max():.3f} rad/s")

## Behavior-tree nodes and coverage
The next cells summarize the nodes recorded in `behaviors` and measure coverage against the tree each run actually loaded, read from that run's `nav2_behaviors.csv`.

In [ ]:
# Nodes are counted from the behavior-tree event stream, not inferred from navigation success.
node_counts = (df_behaviors.dropna(subset=['behavior_name'])
               .groupby('behavior_name', observed=True)
               .size()
               .sort_values(ascending=False)
               .rename('events')
               .to_frame())
node_counts['runs'] = (df_behaviors.dropna(subset=['behavior_name'])
                      .groupby('behavior_name', observed=True)['run_id']
                      .nunique())
display(node_counts)
print(f"Triggered nodes: {len(node_counts)}; behavior events: {int(node_counts['events'].sum())}")

In [ ]:
from pathlib import Path
import pandas as pd

# nav2_bt_tree writes one row per node of the run's ACTUAL tree (status_name
# 'INVALID' when the node never ticked), so per-run coverage needs no XML file and no
# per-test-family mapping: the denominator is whatever tree nav2 loaded for that run
# (navigate_to_pose_w_replanning_and_recovery.xml for the nav_to_pose/waypoint tests,
# navigate_through_poses_w_replanning_and_recovery.xml for test_nav_through_poses).
cov_rows, all_declared, all_reached = [], set(), set()
for csv_path in sorted(Path(DATA_DIR).rglob('nav2_behaviors.csv')):
    df_run = pd.read_csv(csv_path)
    declared = set(df_run['behavior_name'].dropna().astype(str))
    reached = set(df_run.loc[df_run['status_name'] != 'INVALID',
                            'behavior_name'].dropna().astype(str))
    if not declared:
        continue
    cov_rows.append({'run': csv_path.parent.name,
                     'declared': len(declared), 'reached': len(reached),
                     'coverage_pct': 100.0 * len(reached) / len(declared)})
    all_declared |= declared
    all_reached |= reached
if cov_rows:
    cov = pd.DataFrame(cov_rows).set_index('run').sort_values('coverage_pct')
    display(cov)
    print(f"Per-run coverage: min {cov['coverage_pct'].min():.1f}%, "
          f"mean {cov['coverage_pct'].mean():.1f}%, max {cov['coverage_pct'].max():.1f}%")
    print(f"Across {len(cov_rows)} run(s): {len(all_reached)} of {len(all_declared)} distinct "
          f"node labels ticked ({100.0 * len(all_reached) / len(all_declared):.1f}%).")
    missing = sorted(all_declared - all_reached)
    if missing:
        print('Distinct tree labels never observed:')
        print('\n'.join(f'  - {label}' for label in missing))
else:
    print('No nav2_behaviors.csv found under DATA_DIR; coverage needs a completed run.')